In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

# Monitoring training

A Glass Box UMAP fit on a thousand points is over before you can blink. Real workloads aren't always this kind: large neighbor graphs, hundreds of epochs, hyperparameter sweeps that fan out across machines. Once a single fit costs minutes (or hours), you want a way to watch it without sitting in front of a progress bar.

This guide shows how to wire up [TensorBoard](https://www.tensorflow.org/tensorboard) so you can:

- watch the loss curve live, and decide whether to keep going or stop early,
- spot divergence as soon as it appears,
- compare runs side by side when sweeping hyperparameters,
- keep a permanent record of every fit on disk.

All of this is enabled by a single argument: `checkpoint_dir`.

## Setting `checkpoint_dir`

Every fit writes a checkpoint of the best model *and* a TensorBoard event log. By default both go to a temporary directory that is wiped at the end of training. Pass an explicit `checkpoint_dir` and they are kept:

:::{admonition} GlassBoxUMAP API
:class: api, dropdown

From the [API docs](../autoapi/glass_box_umap/index.rst#glass_box_umap.GlassBoxUMAP):

```{eval-rst}
.. autoclass:: glass_box_umap.GlassBoxUMAP
    :noindex:
```
:::

In [ ]:
from pathlib import Path

from glass_box_umap import GlassBoxUMAP

run_dir = Path("runs/digits-baseline")
embedder = GlassBoxUMAP(
    epochs=300,
    random_state=7,
    checkpoint_dir=run_dir,
    quiet=True,
)

## A worked example

We'll use scikit-learn's digits dataset — small enough to run locally in well under a minute, but enough epochs that the loss curve has interesting shape to look at.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

X, _ = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)

embedder.fit(X)

Once the fit completes, `run_dir` contains two things — a checkpoint and a log:

In [ ]:
for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(path)

- **`checkpoints/best.ckpt`** is the same checkpoint that `restore_best_weights` (default `True`) reloads at the end of training, so you don't normally need to touch it. It stays on disk in case you want to inspect or reload a specific run later.
- **`logs/events.out.tfevents.…`** is the TensorBoard event file. This is what we'll point TensorBoard at next.

## Launching the dashboard

`tensorboard` ships as a dependency of `glass-box-umap`, so nothing extra needs to be installed. From the project root:

```bash
tensorboard --logdir runs/
```

That starts a server (default `http://localhost:6006`) which auto-discovers every event file under `runs/`. Leave it running while you train — it polls the directory and refreshes the curves as new events are written, so you can watch a fit in progress.

## What you'll see

Glass Box UMAP logs two scalars during training:

- **`loss_step`** — the UMAP loss at each minibatch. Noisy, but useful for spotting divergence in the first few hundred steps before any epoch has completed.
- **`loss_epoch`** — the mean loss over each epoch. This is the metric the best-checkpoint callback monitors, so it's the one to watch when deciding whether training has converged.

A typical workflow:

1. Start training with a generous `epochs` budget.
2. Open TensorBoard and watch `loss_epoch`.
3. When the curve flattens, kill the run — `best.ckpt` is already on disk.

## Comparing runs

TensorBoard plots every subdirectory of `--logdir` as a separate run, so the easiest way to compare hyperparameters is to give each fit its own subdirectory under a shared parent:

In [ ]:
for n_neighbors in (5, 15, 50):
    GlassBoxUMAP(
        n_neighbors=n_neighbors,
        epochs=300,
        random_state=7,
        checkpoint_dir=Path(f"runs/digits-nn-{n_neighbors}"),
        quiet=True,
    ).fit(X)

Pointing TensorBoard at `runs/` now overlays all three loss curves in the same chart, which makes it easy to see which value of `n_neighbors` converged faster or to a lower loss.

## A glance at the dashboard

Once TensorBoard is up, the **Scalars** tab is the one you'll spend most of your time in. You'll see one line per run for each scalar — `loss_epoch` is the smooth curve to make decisions on, `loss_step` is the noisy one underneath:

```{image} ../_assets/tensorboard_dashboard.png
:alt: TensorBoard Scalars tab showing loss_epoch and loss_step for several Glass Box UMAP runs.
:width: 100%
```